**Joins in Dataframe – Part 1**


In [0]:
# Employee DataFrame

emp_data = [
    (1, "Sham", 101, "Pune"),
    (2, "Rahul", 102, "Mumbai"),
    (3, "Priya", 103, "Bangalore"),
    (4, "Amit", 104, "Chennai"),
    (5, "Sneha", None, "Pune"),
    (6, "Vikas", 105, None),
    (7, "Neha", 101, "Hyderabad"),
    (8, "Rohit", 106, "Delhi"),
    (9, None, 107, "Pune"),
    (10, "Kiran", None, None)
]
emp_cols = ["emp_id", "emp_name", "dept_id", "city"]
emp = spark.createDataFrame(emp_data, emp_cols)

# Department DataFrame

dept_data = [
    (101, "HR", "A"),
    (102, "Finance", "B"),
    (103, "IT", "A"),
    (105, "Marketing", None),
    (107, "Operations", "C"),
    (108, "Sales", "B"),
    (None, "Unknown", "Z")
]
dept_cols = ["dept_id", "dept_name", "dept_group"]
dep = spark.createDataFrame(dept_data, dept_cols)

display(emp)
display(dep)

In [0]:
#ineer join
inner_join  = emp.join(dep, on="dept_id", how="inner")
display(inner_join)

**Explanation:**

- **Purpose:** Returns rows where there is a match in both DataFrames (df1 and df2) based on the common_column.
- **Behavior:** Rows with no matching value in either DataFrame are excluded.
- **Use Case:** When you only need records that exist in both DataFrames.

In [0]:
#left join
left_join = emp.join(dep, on="dept_id", how="left")
display(left_join)

**Explanation:**

- **Purpose:** Returns all rows from df1 and the matching rows from df2. If no match exists in df2, the result will contain NULL for columns from df2.
- **Behavior:** All rows from the left DataFrame (df1) are preserved, even if there’s no match in the right DataFrame (df2).
- **Use Case:** When you want to retain all rows from df1, even if there's no match in df2.

In [0]:
#right join
right_join = emp.join(dep, on = "dept_id", how = "right")
display(right_join)

**Explanation:**

- **Purpose:** Returns all rows from df2 and the matching rows from df1. If no match exists in df1, the result will contain NULL for columns from df1.
- **Behavior:** All rows from the right DataFrame (df2) are preserved, even if there’s no match in the left DataFrame (df1).
- **Use Case:** When you want to retain all rows from df2, even if there's no match in df1.

In [0]:
#full join (outer join)
outer_join = emp.join(dep, on = "dept_id", how = "outer")
display(outer_join)

**Explanation:**

- **Purpose:** Returns all rows when there is a match in either df1 or df2. Non-matching rows will have NULL values in the columns from the other DataFrame.
- **Behavior:** Retains all rows from both DataFrames, filling in NULL where there is no match.
- **Use Case:** When you want to retain all rows from both DataFrames, regardless of whether there’s a match.

In [0]:
#left Semi Join
left_semi_join = emp.join(dep, on = "dept_id", how = "left_semi")
display(left_semi_join)

**Explanation:**

- **Purpose:** Returns only the rows from df1 where there is a match in df2. It behaves like an inner join but only keeps columns from df1.
- **Behavior:** Filters df1 to only keep rows that have a match in df2.
- **Use Case:** When you want to filter df1 to keep rows with matching keys in df2, but you don’t need columns from df2.

In [0]:
#left anti jopn
left_anti_join = emp.join(dep, on = "dept_id", how = "left_anti")
display(left_anti_join)

**Explanation:**

- **Purpose:** Returns only the rows from df1 that do not have a match in df2.
- **Behavior:** Filters out rows from df1 that have a match in df2.
- **Use Case:** When you want to filter df1 to keep rows with no matching keys in df2.

In [0]:
#cross join
cross_join = emp.crossJoin(dep)
display(cross_join)

**Explanation:**

- **Purpose:** Returns the Cartesian product of df1 and df2, meaning every row of df1 is paired with every row of df2.
- **Behavior:** The number of rows in the result will be the product of the row count of df1 and df2.
- **Use Case:** Typically used in edge cases or for generating combinations of rows, but be cautious as it can result in a very large DataFrame.

In [0]:
#Join with Explicit Conditions
explicit = emp.join(dep, (emp["dept_id"] == dep["dept_id"]), "inner")
display(explicit)

In [0]:
#Join with Explicit Conditions (left)
explicit_left = emp.join(dep, (emp["dept_id"] == dep["dept_id"]), "left")
display(explicit_left)

In [0]:
#Join with Explicit Conditions (right)
explicit_right = emp.join(dep, (emp["dept_id"] == dep["dept_id"]), "right")
display(explicit_right)

**Explanation:**

- **Purpose:** This is an example of an inner join where the common columns have different names in df1 and df2.
- **Behavior:** Joins df1 and df2 based on a condition where columnA from df1 matches columnB from df2.
- **Use Case:** When the join condition involves columns with different names or more complex conditions.

**Conclusion:**

- **Inner Join:** Matches rows from both DataFrames.
- **Left/Right Join:** Keeps all rows from the left or right DataFrame and matches where possible.
- **Full Join:** Keeps all rows from both DataFrames.
- **Left Semi:** Filters df1 to rows that match df2 without including columns from df2.
- **Left Anti:** Filters df1 to rows that do not match df2.
- **Cross Join:** Returns the Cartesian product, combining all rows of both DataFrames.
- **Explicit Condition Join:** Allows complex join conditions, including columns with different names.

**Broadcast Join** 
(Optimizing a join with a smaller DataFrame)


In [0]:
from pyspark.sql import functions as f

broadcast_join = emp.join(f.broadcast(dep), "dept_id", "inner" )
broadcast_join.display()

Broadcast Joins in PySpark

- **Definition:** A broadcast join optimizes joins when one DataFrame is small enough to fit into memory by broadcasting it to all nodes. This eliminates the need to shuffle data across the cluster, significantly improving performance for large datasets.
- **Usage:** Recommended for joining a large DataFrame with a small DataFrame (that can fit into memory). You can force a broadcast join using `broadcast(df_small)`.
- **Advantages:**
  - Avoids data shuffling, which can speed up processing for suitable cases.
  - Reduces memory consumption and network I/O for specific types of joins.

**In summary:**
- **Left Anti Join** is useful for identifying non-matching rows from one DataFrame against another.
- **Broadcast Join** is a performance optimization technique ideal for joining a large DataFrame with a small one efficiently, reducing shuffle costs.